In [0]:
import codecs
import xml.etree.ElementTree as ET
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import expr
import zipfile
import os
import yaml
from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql.functions import col, lit, expr, when

In [0]:
def parse_rejections_files_format(_xml_file_path, _txt_file_path):
    """
    This function parses an XML file and a text file according to the XML's format 
    specifications, dynamically determining column positions and schema, 
    and returns a Spark DataFrame.

    Args:
    _xml_file_path (str): The path to the XML file containing the format specification.
    _txt_file_path (str): The path to the text file that will be parsed based on the format.

    Returns:
    _parsed_df (DataFrame): A Spark DataFrame containing the parsed data from the text file.
    """

    with open(_xml_file_path, "rb") as f:
        raw_data = f.read()

    # Detect encoding based on BOM
    if raw_data.startswith(codecs.BOM_UTF16_LE):
        encoding = "utf-16-le"
        raw_data = raw_data[len(codecs.BOM_UTF16_LE):]  # Remove BOM
    elif raw_data.startswith(codecs.BOM_UTF16_BE):
        encoding = "utf-16-be"
        raw_data = raw_data[len(codecs.BOM_UTF16_BE):]  # Remove BOM
    else:
        encoding = "utf-8"  # Default to UTF-8

    # Decode bytes to string
    xml_string = raw_data.decode(encoding)

    # Parse XML safely
    root = ET.fromstring(xml_string)

    print(root)

    print("XML Parsed Successfully!")
    # Extract FIELD elements
    fields = root.find("{http://schemas.microsoft.com/sqlserver/2004/bulkload/format}RECORD").findall("{http://schemas.microsoft.com/sqlserver/2004/bulkload/format}FIELD")

    # Extract COLUMN mappings
    columns = root.find("{http://schemas.microsoft.com/sqlserver/2004/bulkload/format}ROW").findall("{http://schemas.microsoft.com/sqlserver/2004/bulkload/format}COLUMN")

    # Detect Delimiter (Check Last Column Separately)
    general_delimiter = None
    last_column_delimiter = None

    for idx, field in enumerate(fields):
        term = field.attrib.get("TERMINATOR")
        if term:
            term = term.replace("\\r\\n", "\r\n")  # Handle newline cases
            if idx != len(fields) - 1:  # Check if it's the last field
                general_delimiter = term

    if general_delimiter == None:

        initial_file_processing(_txt_file_path)
        # Compute column positions dynamically
        col_positions = []
        start_pos = 0

        for field in fields:
            field_length = int(field.attrib.get("LENGTH", 0))
            col_positions.append((start_pos, start_pos + field_length))
            start_pos += field_length  # Move to next column

        # Extract COLUMN mappings from XML

        # Map column indexes to column names
        col_mapping = {int(col.attrib["SOURCE"]): col.attrib["NAME"] for col in columns}
        df_raw = spark.read.option("encoding", "utf-8").text(_txt_file_path)

        print(col_positions)
        for i, (start, end) in enumerate(col_positions, 1):
            col_name = col_mapping.get(i, f"Col_{i}")
            length = end-start
            if length == 0:
                # In case of the last column, extract everything from 'start' to the end
                df_raw = df_raw.withColumn(col_name, expr(f"substring(value, {start+1}, length(value) - {start})"))
            else:
                # Extract the fixed-length substring
                df_raw = df_raw.withColumn(col_name, expr(f"substring(value, {start+1}, {length})"))

        _parsed_df = df_raw.drop("value")

        return _parsed_df

    print(f"Extracted Delimiter: '{general_delimiter}'")



    # Create Spark Schema dynamically
    schema_fields = [StructField(col.attrib["NAME"], StringType(), True) for col in columns]
    spark_schema = StructType(schema_fields)

    print("Generated Spark Schema:")
    print(spark_schema)

    _parsed_df = spark.read.option("header", "false") \
               .option("delimiter", general_delimiter) \
                .option("inferSchema", "false") \
                .option("encoding", "ISO-8859-7") \
                .csv(_txt_file_path, schema=spark_schema)


    return _parsed_df

In [0]:
def initial_file_processing(_file_path):
    """
    This function processes a file by checking its encoding and converting it to UTF-8 if needed.
    
    The function first attempts to read the file in UTF-8 encoding. If the file cannot be read as UTF-8 
    (due to a `UnicodeDecodeError`), it assumes the file is in a different encoding (e.g., Windows-1253, 
    commonly used for Greek text) and tries reading it with that encoding. If successful, the content is 
    then rewritten in UTF-8 encoding.

    If the file is already in UTF-8, the function will skip the conversion and print a message indicating 
    that no processing is needed.

    Args:
    _file_path (str): The path to the file that needs to be processed.

    Returns:
    None: The function modifies the file in place if needed, and prints status messages.
    """
    try:
        # Try to open the file with UTF-8 encoding first
        with open(_file_path, 'r', encoding='utf-8') as infile:
            content = infile.read()
        print(f"The file is already in UTF-8 encoding. No processing needed.")
        return
    except UnicodeDecodeError:
        print(f"File is not in UTF-8 encoding.")
        try:
            with open(_file_path, 'r', encoding='windows-1253') as infile:
                content = infile.read()
        except UnicodeDecodeError as e:
            """
            If a UnicodeDecodeError occurs again, it indicates that the file can't be decoded 
            using the Windows-1253 encoding either.
            """
            print(f"Error: The file could not be decoded with Windows-1253 encoding.")
            print(f"Details: {e}")
            return
        except Exception as e:
            """
            Handle any other unexpected errors (e.g., file not found, permission issues).
            """
            print(f"An unexpected error occurred while reading the file.")
            print(f"Details: {e}")
            return

        # If the file was successfully read using Windows-1253 encoding, write the content back in UTF-8
        try:
            with open(_file_path, 'w', encoding='utf-8') as outfile:
                outfile.write(content)  # Write the content in UTF-8 encoding
            print(f"File successfully converted to UTF-8 encoding.")
        except Exception as e:
            """
            If an error occurs during the writing process (e.g., permissions, I/O errors), 
            it will be caught here.
            """
            print(f"Error while writing the file in UTF-8 encoding.")
            print(f"Details: {e}")



In [0]:
def unzip_files(_zip_files_path, _extract_files_path):
    """ 
    This function unzips all the files in the specified directory (_zip_files_path) and extracts them 
    to the provided extraction path (_extract_files_path).
    
    Args:
    _zip_files_path (str): The path to the directory containing the zip files.
    _extract_files_path (str): The path where the zip files should be extracted.
    """
    # List all the files in the provided zip files directory (_zip_files_path)
    _zip_files = dbutils.fs.ls(_zip_files_path)
    
    # Loop over each file found in the directory
    for _file in _zip_files:
        # Construct the full path to the current zip file by joining the directory path and file name
        current_zip_path = f"{_zip_files_path}/{_file.name}"
        
        # Open the zip file in read mode ('r')
        with zipfile.ZipFile(current_zip_path, "r") as zip_ref:
            # Extract all the contents of the zip file to the provided extraction path (_extract_files_path)
            zip_ref.extractall(_extract_files_path)
            
            # Print a message confirming that the zip file was successfully extracted
            print(f"Unzipped file: {current_zip_path}")

In [0]:
def load_config(config_path="tiresias_config.yaml"):
    """Loads the YAML configuration file."""
    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

In [0]:
def evaluate_conditions(file_name, conditions):
    """
    Evaluates conditions dynamically based on file name attributes.

    Args:
        file_name (str): The name of the file.
        conditions (list/dict): The conditions to evaluate.

    Returns:
        int: The evaluated value.
    """
    if isinstance(conditions, dict):  # Single condition (e.g., "contains": "MME")
        if "contains" in conditions and conditions["contains"] in file_name:
            return 1
        if "startswith" in conditions and file_name.startswith(conditions["startswith"]):
            return 1
        return 0
    
    elif isinstance(conditions, list):  # List of conditions for non_updated_type
        print(conditions)
        for condition in conditions:
            if ("contains" not in condition or condition["contains"] in file_name) and \
            ("startswith" not in condition or file_name.startswith(condition["startswith"])):
                print(condition)
                return condition["value"]
        return 0

In [0]:
def determine_format_file(attributes, format_rules):
    """
    Determines the file format based on dynamic conditions.

    Args:
        attributes (dict): File attributes (e.g., is_business, is_rejection).
        format_rules (dict): Mapping of conditions to format files.

    Returns:
        str or None: The corresponding format file.
    """
    for condition, format_file in format_rules.items():

        print(format_file)
        # Safely evaluate condition by using `attributes` as the local context
        try:
            # Ensure that the condition is using the correct variable names (e.g., snake_case).
            condition_expr = condition
            for key, value in attributes.items():
                # Replace keys like 'is_business' in the condition string with actual values from attributes
                print(key)
                print(value)
                condition_expr = condition_expr.replace(key, str(value))
 
            print(f"Evaluating Expression: {condition_expr}")  # Debugging: Show the final expression being evaluated

            # Safely evaluate the condition with updated variables
            if eval(condition_expr):
                print('test')
                return format_file
        except Exception as e:
            print(f"Error evaluating condition '{condition}': {e}")  # Debugging: Show errors

    return None


In [0]:
def file_parameters(_file_name, config):
    """
    Dynamically determines parameters for a processed file.

    Args:
        _file_name (str): The name of the file.
        config (dict): Configuration dictionary.

    Returns:
        tuple: (_format_file, _is_visually_checked, _is_business, _is_rejection, _non_updated_type)
    """
    attributes = {
        key: evaluate_conditions(_file_name, rule)
        for key, rule in config["attributes"].items()
    }

    _format_file = determine_format_file(attributes, config["file_formats"])

    return  _format_file, \
            attributes["is_visually_checked"], \
            attributes["is_business"], \
            attributes["is_rejection"], \
            attributes["non_updated_type"]
    